# ADM ImageNet-256 Metrics Evaluation

Use OpenAI's ADM evaluation implementation to compare a generated sample batch with `VIRTUAL_imagenet256_labeled.npz`.

**Outputs:** FID, sFID, Inception Score, precision, and recall.

**Real-data input:** only the ADM reference `.npz` is required; raw ImageNet folders are not used.

**Generated-data input:** an `.npz` containing `arr_0` with shape `[N, 256, 256, 3]` and `uint8` values in `[0, 255]`. For the standard benchmark, use 50,000 generated images.


## Outline

1. Configure paths.
2. Check dependencies.
3. Acquire the official ADM evaluator and, optionally, the reference batch.
4. Validate both `.npz` files without loading their image arrays into memory.
5. Run the evaluator and save the metrics as JSON.


## 1. Configure paths

Change `SAMPLES_NPZ` to the `.npz` produced by `sample_ddp.py`. Set `DOWNLOAD_REFERENCE = True` only if the approximately 2 GB ADM reference file is not already available.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import re
import shlex
import subprocess
import sys
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

REFERENCE_NPZ = PROJECT_ROOT / "VIRTUAL_imagenet256_labeled.npz"
SAMPLES_NPZ = PROJECT_ROOT / "samples_50000.npz"  # Change this path.
EVALUATOR_PATH = PROJECT_ROOT / "tools" / "adm_evaluation" / "evaluator.py"
RESULTS_JSON = PROJECT_ROOT / "adm_imagenet256_metrics.json"

DOWNLOAD_REFERENCE = False
SAVE_RESULTS = True

REFERENCE_URL = (
    "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/"
    "ref_batches/imagenet/256/VIRTUAL_imagenet256_labeled.npz"
)
EVALUATOR_URL = (
    "https://raw.githubusercontent.com/openai/guided-diffusion/"
    "main/evaluations/evaluator.py"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Reference:    {REFERENCE_NPZ}")
print(f"Samples:      {SAMPLES_NPZ}")


## 2. Check dependencies

The official evaluator uses TensorFlow's compatibility API plus SciPy, Requests, and tqdm. Install missing packages in the active notebook kernel, then restart the kernel:

```python
%pip install tensorflow scipy requests tqdm
```


In [ ]:
required_modules = {
    "numpy": "numpy",
    "scipy": "scipy",
    "requests": "requests",
    "tqdm": "tqdm",
    "tensorflow": "tensorflow",
}

missing_packages = [
    package
    for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    raise RuntimeError(
        "Missing packages: "
        + ", ".join(missing_packages)
        + ". Install them with: %pip install tensorflow scipy requests tqdm"
    )

print("All ADM evaluator dependencies are available.")


## 3. Acquire the official evaluation assets

The next cell downloads OpenAI's evaluator when it is missing. It also applies the minimal `np.bool` compatibility fix required by modern NumPy versions. The Inception graph is downloaded automatically by the evaluator on its first run.


In [ ]:
from tqdm.auto import tqdm


def download_file(url: str, destination: Path, chunk_size: int = 8 * 1024 * 1024) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")

    with urllib.request.urlopen(url) as response:
        total = int(response.headers.get("Content-Length", 0)) or None
        with temporary.open("wb") as output, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as progress:
            while True:
                chunk = response.read(chunk_size)
                if not chunk:
                    break
                output.write(chunk)
                progress.update(len(chunk))

    temporary.replace(destination)


if not EVALUATOR_PATH.exists():
    download_file(EVALUATOR_URL, EVALUATOR_PATH)

# OpenAI's original script uses the removed np.bool alias in precision/recall.
evaluator_source = EVALUATOR_PATH.read_text(encoding="utf-8")
compatible_source = evaluator_source.replace("dtype=np.bool", "dtype=bool")
if compatible_source != evaluator_source:
    EVALUATOR_PATH.write_text(compatible_source, encoding="utf-8")
    print("Applied modern NumPy compatibility fix to evaluator.py.")

if not REFERENCE_NPZ.exists() and DOWNLOAD_REFERENCE:
    download_file(REFERENCE_URL, REFERENCE_NPZ)

if REFERENCE_NPZ.exists():
    print(f"Reference batch ready: {REFERENCE_NPZ}")
else:
    print("Reference batch is missing. Place it at REFERENCE_NPZ or set DOWNLOAD_REFERENCE=True.")

print(f"Evaluator ready: {EVALUATOR_PATH}")


## 4. Validate the NPZ inputs

This reads only the embedded NumPy headers, so it does not decompress the multi-gigabyte image arrays. The reference must include ADM's precomputed FID and sFID statistics. The generated batch must include an `arr_0` image array.


In [ ]:
def inspect_npz_headers(path: Path) -> dict[str, dict[str, object]]:
    if not path.exists():
        raise FileNotFoundError(path)

    arrays: dict[str, dict[str, object]] = {}
    with zipfile.ZipFile(path) as archive:
        for member in archive.namelist():
            if not member.endswith(".npy"):
                continue
            with archive.open(member) as stream:
                version = np.lib.format.read_magic(stream)
                if version == (1, 0):
                    shape, fortran_order, dtype = np.lib.format.read_array_header_1_0(stream)
                else:
                    shape, fortran_order, dtype = np.lib.format.read_array_header_2_0(stream)
            arrays[Path(member).stem] = {
                "shape": tuple(shape),
                "dtype": str(dtype),
                "fortran_order": bool(fortran_order),
            }
    return arrays


reference_arrays = inspect_npz_headers(REFERENCE_NPZ)
sample_arrays = inspect_npz_headers(SAMPLES_NPZ)

required_reference_keys = {"arr_0", "mu", "sigma", "mu_s", "sigma_s"}
missing_reference_keys = required_reference_keys - set(reference_arrays)
if missing_reference_keys:
    raise ValueError(
        f"Reference batch is missing required arrays: {sorted(missing_reference_keys)}"
    )

if "arr_0" not in sample_arrays:
    raise ValueError("Generated sample NPZ must contain an arr_0 image array.")

sample_info = sample_arrays["arr_0"]
sample_shape = sample_info["shape"]
sample_dtype = np.dtype(sample_info["dtype"])

if len(sample_shape) != 4 or tuple(sample_shape[1:]) != (256, 256, 3):
    raise ValueError(
        f"Expected generated arr_0 shape [N, 256, 256, 3], found {sample_shape}."
    )
if sample_dtype != np.dtype("uint8"):
    raise ValueError(f"Expected generated arr_0 dtype uint8, found {sample_dtype}.")
if sample_shape[0] != 50_000:
    warnings.warn(
        f"Generated batch contains {sample_shape[0]:,} images, not the standard 50,000. "
        "Label the result with the actual sample count and do not compare it directly with FID-50K."
    )

print("Reference arrays:")
for name in sorted(reference_arrays):
    print(f"  {name:8s} {reference_arrays[name]}")

print("\nGenerated arrays:")
for name in sorted(sample_arrays):
    print(f"  {name:8s} {sample_arrays[name]}")

print("\nInput validation passed.")


## 5. Run ADM evaluation

The evaluator streams progress to the notebook. On first use it downloads the approximately 100 MB ADM Inception graph. Runtime and memory usage depend on the machine; evaluating 50,000 images can take substantial time.


In [ ]:
command = [
    sys.executable,
    str(EVALUATOR_PATH),
    str(REFERENCE_NPZ),
    str(SAMPLES_NPZ),
]

print("Running:", shlex.join(command))
print()

process = subprocess.Popen(
    command,
    cwd=EVALUATOR_PATH.parent,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

output_lines: list[str] = []
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
    output_lines.append(line)

return_code = process.wait()
raw_output = "".join(output_lines)
if return_code != 0:
    raise RuntimeError(f"ADM evaluator failed with exit code {return_code}.")


## 6. Parse and save results

FID, sFID, precision, and recall compare your generated batch with the ADM reference. Inception Score is computed from the generated batch alone. Lower FID/sFID and higher IS/precision/recall are better, but comparisons are valid only when preprocessing and sampling protocols match.


In [ ]:
metric_patterns = {
    "inception_score": r"(?m)^Inception Score:\s*([-+0-9.eE]+)",
    "fid": r"(?m)^FID:\s*([-+0-9.eE]+)",
    "sfid": r"(?m)^sFID:\s*([-+0-9.eE]+)",
    "precision": r"(?m)^Precision:\s*([-+0-9.eE]+)",
    "recall": r"(?m)^Recall:\s*([-+0-9.eE]+)",
}

metrics: dict[str, float] = {}
for name, pattern in metric_patterns.items():
    match = re.search(pattern, raw_output)
    if match is None:
        raise RuntimeError(f"Could not parse {name} from evaluator output.")
    metrics[name] = float(match.group(1))

print("\nADM ImageNet-256 metrics")
print(f"  FID:             {metrics['fid']:.6f}")
print(f"  sFID:            {metrics['sfid']:.6f}")
print(f"  Inception Score: {metrics['inception_score']:.6f}")
print(f"  Precision:       {metrics['precision']:.6f}")
print(f"  Recall:          {metrics['recall']:.6f}")

result_record = {
    "reference_npz": str(REFERENCE_NPZ),
    "samples_npz": str(SAMPLES_NPZ),
    "num_generated_images": int(sample_shape[0]),
    "metrics": metrics,
}

if SAVE_RESULTS:
    RESULTS_JSON.write_text(json.dumps(result_record, indent=2) + "\n", encoding="utf-8")
    print(f"\nSaved: {RESULTS_JSON}")

result_record


## Reporting and pitfalls

- Report this result as **ADM ImageNet-256 FID-50K** only when the generated batch contains 50,000 images and follows the same sampling protocol as the compared methods.
- Keep image resolution, VAE decoder, sampling steps, guidance scale, precision, and class-label sampling fixed across model comparisons.
- The ADM evaluator computes pooled distribution metrics. It does **not** compute 1,000 separate class-wise FIDs, even though the reference filename contains `labeled`.
- The evaluator does not verify that your generated labels are balanced. Validate class counts during sample generation when exact balance is part of your protocol.
- Do not compare these scores directly with `clean-fid` scores unless the other work explicitly uses the same implementation and preprocessing.


## Reproducibility check

Run the evaluation twice with the same `.npz` files. The parsed metrics should be identical. If they differ, record the TensorFlow, NumPy, SciPy, CUDA, and driver versions before comparing model variants.
